# Week 3 - Customer Intelligence System

## Objective

The goal of this project is to build a simple Customer Intelligence System using classification, ensemble learning and clustering.

Dataset used: Country data from Kaggle.

Main tasks:
- Data cleaning
- EDA
- Classification
- Random Forest
- XGBoost
- K-Means clustering
- DBSCAN clustering
- Customer / country segmentation insights

This notebook is written in a simple learning-student style.

In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "1"

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

from xgboost import XGBClassifier

RANDOM_STATE = 42


## Dataset Loading

The dataset should be downloaded from Kaggle and placed in the project folder.

Expected file name: `Country-data.csv`

In [ ]:
possible_paths = [
    Path("Country-data.csv"),
    Path("country-data.csv"),
    Path("data/Country-data.csv"),
    Path("data/country-data.csv")
]

data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("Country-data.csv not found. Please keep the real Kaggle country data file in this folder.")

df = pd.read_csv(data_path)

print("Dataset loaded from:", data_path)
print("Shape:", df.shape)
df.head()


In [ ]:
assert df.shape[0] > 0, "Dataset is empty"
assert df.shape[1] > 0, "No columns found"
print("Dataset loaded successfully.")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## Data Cleaning

Here I removed duplicates and handled missing values.

In [ ]:
df.columns = df.columns.str.strip()
df = df.drop_duplicates()

for col in df.columns:
    if col != "country":
        df[col] = pd.to_numeric(df[col], errors="coerce")

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print("Cleaned shape:", df.shape)
df.head()

## Exploratory Data Analysis

EDA helps to understand the dataset before building models.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["gdpp"], bins=25, kde=True)
plt.title("Distribution of GDP per Capita")
plt.xlabel("GDPP")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="income", y="gdpp")
plt.title("Income vs GDPP")
plt.xlabel("Income")
plt.ylabel("GDPP")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="child_mort", y="life_expec")
plt.title("Child Mortality vs Life Expectancy")
plt.xlabel("Child Mortality")
plt.ylabel("Life Expectancy")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

## EDA Findings

Countries with higher income and GDPP generally look more developed.

Countries with high child mortality usually have lower life expectancy.

These patterns can help in classification and segmentation.

## Creating a Classification Target

The dataset is mainly for unsupervised learning, so I created a simple target column.

`High_Need` = 1 if a country has high child mortality, low income and low GDPP.

This target is used only for learning classification models.

In [ ]:
child_limit = df["child_mort"].median()
income_limit = df["income"].median()
gdpp_limit = df["gdpp"].median()

df["High_Need"] = (
    (df["child_mort"] > child_limit) &
    (df["income"] < income_limit) &
    (df["gdpp"] < gdpp_limit)
).astype(int)

df["High_Need"].value_counts()

In [ ]:
features = [
    "child_mort", "exports", "health", "imports", "income",
    "inflation", "life_expec", "total_fer", "gdpp"
]

X = df[features]
y = df["High_Need"]

## Train-Test Split

I used train-test split for classification.

In [ ]:
stratify_value = y if y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=stratify_value
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Classification Models

I trained two ensemble classification models:

- Random Forest
- XGBoost


In [ ]:
def get_scores(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0)
    }

In [ ]:
# Direct Random Forest and XGBoost classifiers
rf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1))
])

xgb_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=1
    ))
])

models = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}


In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    
    row = {"Model": name}
    row.update(get_scores(y_test, pred))
    results.append(row)
    
    trained_models[name] = model

results_df = pd.DataFrame(results)
results_df

In [ ]:
best_model_name = results_df.sort_values("F1 Score", ascending=False).iloc[0]["Model"]
best_model = trained_models[best_model_name]

best_pred = best_model.predict(X_test)

print("Best Model:", best_model_name)
print(classification_report(y_test, best_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Hyperparameter Tuning

I used GridSearchCV to improve the Random Forest model.

In [ ]:
rf_grid = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1))
    ]),
    {
        "model__n_estimators": [50, 100],
        "model__max_depth": [3, 5, None]
    },
    cv=3,
    scoring="f1"
)

rf_grid.fit(X_train, y_train)

print("Best RF Params:", rf_grid.best_params_)
print("Best CV F1:", rf_grid.best_score_)

In [ ]:
tuned_rf = rf_grid.best_estimator_
tuned_pred = tuned_rf.predict(X_test)

tuned_scores = get_scores(y_test, tuned_pred)
tuned_scores

## K-Means Clustering

K-Means is used to segment countries/customers into groups.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

inertia = []
cluster_range = range(1, 8)

for k in cluster_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(cluster_range, inertia, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
df["KMeans_Cluster"] = kmeans.fit_predict(X_scaled)

df["KMeans_Cluster"].value_counts()

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_data = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 5))
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=df["KMeans_Cluster"], palette="Set2")
plt.title("K-Means Clusters using PCA")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

In [ ]:
cluster_profile = df.groupby("KMeans_Cluster")[features].mean()
cluster_profile

## DBSCAN Clustering

DBSCAN is another clustering method. It can also mark some points as noise using cluster label `-1`.

In [ ]:
dbscan = DBSCAN(eps=2.0, min_samples=4)
df["DBSCAN_Cluster"] = dbscan.fit_predict(X_scaled)

df["DBSCAN_Cluster"].value_counts()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=df["DBSCAN_Cluster"], palette="Set1")
plt.title("DBSCAN Clusters using PCA")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

## Customer Segmentation Insights

Here I give simple names to K-Means clusters using average GDPP.

In [ ]:
gdpp_order = cluster_profile["gdpp"].sort_values().index.tolist()

segment_names = {
    gdpp_order[0]: "High Need Segment",
    gdpp_order[1]: "Developing Segment",
    gdpp_order[2]: "Developed Segment"
}

df["Segment"] = df["KMeans_Cluster"].map(segment_names)

df[["country", "KMeans_Cluster", "Segment", "High_Need"]].head(10)

In [ ]:
segment_summary = df.groupby("Segment")[features + ["High_Need"]].mean()
segment_summary

## Final Insights

- Random Forest and XGBoost/Boosting were used for classification.
- K-Means created clear customer/country segments.
- DBSCAN helped identify dense groups and possible noise points.
- High Need Segment has lower GDPP/income and higher child mortality.
- Developed Segment has higher income, higher GDPP and better life expectancy.

This system can help identify which customer/country groups need more attention.

# Conclusion

An end-to-end Customer Intelligence System was built.

The project covered:
- Data cleaning
- EDA
- Classification
- Ensemble learning
- Random Forest
- XGBoost
- Hyperparameter tuning
- K-Means clustering
- DBSCAN clustering
- Actionable segmentation insights